## Re-parse particle-contaminated author_names rows

One-off backfill. `CreateAuthorNames` is incremental (`LEFT ANTI JOIN` against existing `author_names`), so the new `_strip_surname_particles` logic doesn't retroactively apply to the ~222M rows already in the table. This notebook deletes the rows whose OLD parse left a leading surname particle (`de`, `da`, `van der`, etc.) in `last` so the next `CreateAuthorNames` run re-parses them with the particle strip in place.

**Scope:** ~3.98M distinct raw_author_name rows (1.8% of 222M). Confirmed 2026-04-22.

**Sequence:**
1. Verify count in Cell 2
2. Spot-check bad parses in Cell 3
3. Run Cell 4 (DELETE)
4. Run the `CreateAuthorNames` job — it will re-parse the deleted raws via the new logic
5. Re-run Cell 5 to verify particles are cleared
6. Re-run `CreateAuthors` to refresh `authors_for_matching` / downstream aggregates

**Not in scope:** updating existing `work_authors.author_id` assignments. Those were made by MatchAuthors at ingest time and aren't directly tied to `author_names.parsed_name`. The re-parse improves diagnostic queries and future matches; existing assignments stay until a Phase 1 / Phase 2 re-run on the newly-visible candidate pairs.

**Safety:** Delta `DELETE` only; rows will be re-created by the next `CreateAuthorNames` run. Particle list matches `_STRIP_PARTICLES_SINGLE` + `_STRIP_PARTICLES_TWO` in `CreateAuthorNames` cell 4. See `oxjobs/working/split-authors-name-parser/PLAN-surname-particle-parser-fix.md`.

### Cell 1: Affected row count by particle class

In [ ]:
SELECT
  SUM(CASE WHEN parsed_name.last RLIKE '^(de |da |do |dos |das |del )' THEN 1 ELSE 0 END) AS iberian_single,
  SUM(CASE WHEN parsed_name.last RLIKE '^(van |von |zu )' THEN 1 ELSE 0 END) AS germanic_single,
  SUM(CASE WHEN parsed_name.last IN ('de la','de las','de los') OR parsed_name.last RLIKE '^(de la |de las |de los )' THEN 1 ELSE 0 END) AS iberian_compound,
  SUM(CASE WHEN parsed_name.last IN ('van de','van der','van den') OR parsed_name.last RLIKE '^(van de |van der |van den )' THEN 1 ELSE 0 END) AS dutch_compound,
  COUNT(*) AS total_rows
FROM openalex.authors.author_names

### Cell 2: Total affected (union) — the row count that will be deleted

In [ ]:
SELECT COUNT(*) AS rows_to_delete
FROM openalex.authors.author_names
WHERE parsed_name.last RLIKE '^(de |da |do |dos |das |del |van |von |zu )'
   OR parsed_name.last IN ('de la','de las','de los','van de','van der','van den')

### Cell 3: Spot-check — 30 current particle-leading parses (save these raws to verify post-rerun)

In [ ]:
SELECT raw_author_name,
       parsed_name.first AS first,
       parsed_name.middle AS middle,
       parsed_name.last AS last
FROM openalex.authors.author_names
WHERE parsed_name.last RLIKE '^(de |da |do |dos |das |del |van |von |zu )'
   OR parsed_name.last IN ('de la','de las','de los','van de','van der','van den')
ORDER BY RAND()
LIMIT 30

### Cell 4: Delete the particle-contaminated rows

**WARNING**: destructive. Run Cells 1–3 first and confirm scope looks right.

Delta `DELETE` — rows will be re-created by the next `CreateAuthorNames` run via its `LEFT ANTI JOIN` against `locations_mapped` / `openalex_authors` / `work_authors`.

In [ ]:
DELETE FROM openalex.authors.author_names
WHERE parsed_name.last RLIKE '^(de |da |do |dos |das |del |van |von |zu )'
   OR parsed_name.last IN ('de la','de las','de los','van de','van der','van den')

### Cell 5: Post-reparse verification

Run AFTER the next `CreateAuthorNames` job completes. Expect 0 rows — particles should be stripped by the new logic.

**Next steps after this returns 0:**
1. Re-run `CreateAuthors` to refresh `authors_for_matching`
2. MatchAuthors picks up new `last` on the null reservoir automatically on its next run
3. Re-run `SplitOvermergedAuthorsPhase1` → scoped re-match → `SplitOvermergedAuthorsPhase2` → scoped re-match
4. Fresh gold-standard sample, compare scorecard against `2026-04-21_6ca67e28ec62.json`

In [ ]:
SELECT COUNT(*) AS remaining_particle_prefixed
FROM openalex.authors.author_names
WHERE parsed_name.last RLIKE '^(de |da |do |dos |das |del |van |von |zu )'
   OR parsed_name.last IN ('de la','de las','de los','van de','van der','van den')